# Glassdoor India Software Salaries — End-to-End PySpark ETL

**Dataset columns:** `Rating | Company Name | Job Title | Salary (INR) | Salaries Reported | Location | Employment Status | Job Roles`

**Key insight:** Salary is in raw Indian Rupees — we convert to LPA (Lakhs Per Annum) which is the industry standard in India.

**What we build:**
```
glassdoor_salary.csv  (fact table)
city_dim              (Tier 1/2/3 cities + region)  — generated in code
role_dim              (job role → category mapping)  — generated in code
        ↓
Stage 1:  Ingest + profile      (schema, nulls, distributions)
Stage 2:  Clean                 (rename to snake_case, LPA conversion,
                                 dedup, outlier removal, validation)
Stage 3:  Transform             (salary_band, rating_band, title_seniority,
                                 is_reliable, weighted_salary)
Stage 4:  Aggregate             (by role, company, city, seniority)
Stage 5:  Joins                 (city dim + role dim, anti-join quality check)
Stage 6:  Window functions      (rank in role, rank in city, pct of max)
Stage 7:  Performance           (cache, repartition, coalesce, explain)
Stage 8:  Storage               (Parquet partitioned + CSV summary)
Stage 9:  Spark SQL             (5 business queries with CTEs + window SQL)
Stage 10: Full pipeline summary
```


## Stage 0 — Spark Session


In [99]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField,
    StringType, IntegerType, DoubleType
)
from pyspark.sql.window import Window
import os

spark = (
    SparkSession.builder
    .appName("SalaryETLPipeline")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.driver.memory", "2g")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print(f"Spark {spark.version} — session ready")


Spark 4.1.1 — session ready


---
## Stage 1 — Ingestion & Profiling

**Always profile before cleaning.** The schema output showed us:
- Salary is `integer` (raw INR, needs conversion to LPA)
- Column names have spaces (`Company Name`) — must rename to `snake_case`
- `Rating` is `double` — good, no cast needed
- We need to check for nulls and distribution of categorical columns


In [100]:
# ── Load the fact table ──────────────────────────────────
# header=True     : row 1 = column names
# inferSchema     : Spark samples data to auto-detect types
# nullValue="NA"  : treat literal string "NA" as proper null
#                   (very common in Kaggle datasets)

df_raw = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .option("nullValue", "NA")
    .csv("E:/Azur/part_1/My_Project/archive/Salary_Dataset_with_Extra_Features.csv")   # ← update path if needed
)

total_rows = df_raw.count()
print(f"Loaded: {total_rows:,} rows × {len(df_raw.columns)} columns")


Loaded: 22,770 rows × 8 columns


In [101]:
# Schema — always check types before any computation
df_raw.printSchema()


root
 |-- Rating: double (nullable = true)
 |-- Company Name: string (nullable = true)
 |-- Job Title: string (nullable = true)
 |-- Salary: integer (nullable = true)
 |-- Salaries Reported: integer (nullable = true)
 |-- Location: string (nullable = true)
 |-- Employment Status: string (nullable = true)
 |-- Job Roles: string (nullable = true)



In [102]:
# Sample rows — truncate=False shows full values
df_raw.show(5, truncate=False)


+------+--------------------------------+-----------------+-------+-----------------+---------+-----------------+---------+
|Rating|Company Name                    |Job Title        |Salary |Salaries Reported|Location |Employment Status|Job Roles|
+------+--------------------------------+-----------------+-------+-----------------+---------+-----------------+---------+
|3.8   |Sasken                          |Android Developer|400000 |3                |Bangalore|Full Time        |Android  |
|4.5   |Advanced Millennium Technologies|Android Developer|400000 |3                |Bangalore|Full Time        |Android  |
|4.0   |Unacademy                       |Android Developer|1000000|3                |Bangalore|Full Time        |Android  |
|3.8   |SnapBizz Cloudtech              |Android Developer|300000 |3                |Bangalore|Full Time        |Android  |
|4.4   |Appoids Tech Solutions          |Android Developer|600000 |3                |Bangalore|Full Time        |Android  |
+------+

In [103]:
# NULL AUDIT 
# Count nulls per column as a percentage of total rows
# Why %? Raw counts mean nothing without context.
# 500 nulls in 22,700 rows = 2.2% — probably fillable
# 15,000 nulls in 22,700 rows = 66% — probably drop column

print("=== NULL % PER COLUMN ===")
df_raw.select([
    F.round(
        F.count(F.when(F.col(c).isNull(), c)) / total_rows * 100, 2
    ).alias(c)
    for c in df_raw.columns
]).show()


=== NULL % PER COLUMN ===
+------+------------+---------+------+-----------------+--------+-----------------+---------+
|Rating|Company Name|Job Title|Salary|Salaries Reported|Location|Employment Status|Job Roles|
+------+------------+---------+------+-----------------+--------+-----------------+---------+
|   0.0|         0.0|      0.0|   0.0|              0.0|     0.0|              0.0|      0.0|
+------+------------+---------+------+-----------------+--------+-----------------+---------+



In [104]:
#  DISTINCT VALUE COUNTS 
# High distinct count = likely a continuous / ID column
# Low distinct count = likely categorical (good for groupBy)
print("=== DISTINCT VALUE COUNTS ===")
for c in df_raw.columns:
    n = df_raw.select(c).distinct().count()
    print(f"  {c:<25}: {n:>5} distinct values")


=== DISTINCT VALUE COUNTS ===
  Rating                   :    41 distinct values
  Company Name             : 11261 distinct values
  Job Title                :  1080 distinct values
  Salary                   :   316 distinct values
  Salaries Reported        :    82 distinct values
  Location                 :    10 distinct values
  Employment Status        :     4 distinct values
  Job Roles                :    11 distinct values


In [105]:
# Check for duplicates — ideally should be zero
df_raw.printSchema()
df_raw.show(3, truncate=False)
print("Columns:", df_raw.columns)

root
 |-- Rating: double (nullable = true)
 |-- Company Name: string (nullable = true)
 |-- Job Title: string (nullable = true)
 |-- Salary: integer (nullable = true)
 |-- Salaries Reported: integer (nullable = true)
 |-- Location: string (nullable = true)
 |-- Employment Status: string (nullable = true)
 |-- Job Roles: string (nullable = true)

+------+--------------------------------+-----------------+-------+-----------------+---------+-----------------+---------+
|Rating|Company Name                    |Job Title        |Salary |Salaries Reported|Location |Employment Status|Job Roles|
+------+--------------------------------+-----------------+-------+-----------------+---------+-----------------+---------+
|3.8   |Sasken                          |Android Developer|400000 |3                |Bangalore|Full Time        |Android  |
|4.5   |Advanced Millennium Technologies|Android Developer|400000 |3                |Bangalore|Full Time        |Android  |
|4.0   |Unacademy               

In [106]:
# Key distributions — using ACTUAL column names with spaces
print("Job Roles distribution:")
df_raw.groupBy("Job Roles").count().orderBy("count", ascending=False).show(15)

print("Employment Status distribution:")
df_raw.groupBy("Employment Status").count().orderBy("count", ascending=False).show()

print("Location — top 15:")
df_raw.groupBy("Location").count().orderBy("count", ascending=False).show(15)

print("Salary range (raw INR):")
df_raw.select(
    F.min("Salary").alias("min_inr"),
    F.round(F.avg("Salary"),0).alias("avg_inr"),
    F.max("Salary").alias("max_inr")
).show()


Job Roles distribution:
+---------+-----+
|Job Roles|count|
+---------+-----+
|      SDE| 8183|
|  Android| 2945|
| Frontend| 2163|
|     Java| 1858|
|  Testing| 1740|
|      IOS| 1631|
|  Backend| 1194|
|      Web|  999|
|   Python|  947|
| Database|  865|
|   Mobile|  245|
+---------+-----+

Employment Status distribution:
+-----------------+-----+
|Employment Status|count|
+-----------------+-----+
|        Full Time|20083|
|           Intern| 2106|
|       Contractor|  548|
|          Trainee|   33|
+-----------------+-----+

Location — top 15:
+--------------+-----+
|      Location|count|
+--------------+-----+
|     Bangalore| 8264|
|     Hyderabad| 4467|
|     New Delhi| 4176|
|       Chennai| 2458|
|          Pune| 2134|
|        Mumbai|  749|
|       Kolkata|  178|
|Madhya Pradesh|  155|
|        Kerala|  108|
|        Jaipur|   81|
+--------------+-----+

Salary range (raw INR):
+-------+--------+--------+
|min_inr| avg_inr| max_inr|
+-------+--------+--------+
|   2112|69538

---
## Stage 1B — Dimension Tables (Star Schema)

**Two dimension tables:**
1. `city_dim` — maps city names to Tier (1/2/3), region, and state
2. `role_dim` — maps job role strings to category and tech/non-tech flag

These enrich the fact table without storing redundant text in every row.


In [107]:
# DIM 1: City dimension
city_schema = StructType([
    StructField("city",      StringType(), False),
    StructField("city_tier", StringType(), False),
    StructField("region",    StringType(), False),
    StructField("state",     StringType(), False),
])
city_data = [
    ("Bangalore","Tier 1","South","Karnataka"),
    ("Mumbai","Tier 1","West","Maharashtra"),
    ("Delhi","Tier 1","North","Delhi"),
    ("Hyderabad","Tier 1","South","Telangana"),
    ("Chennai","Tier 1","South","Tamil Nadu"),
    ("Pune","Tier 1","West","Maharashtra"),
    ("Kolkata","Tier 1","East","West Bengal"),
    ("Ahmedabad","Tier 2","West","Gujarat"),
    ("Noida","Tier 2","North","Uttar Pradesh"),
    ("Gurgaon","Tier 2","North","Haryana"),
    ("Jaipur","Tier 2","North","Rajasthan"),
    ("Indore","Tier 2","Central","Madhya Pradesh"),
    ("Coimbatore","Tier 2","South","Tamil Nadu"),
    ("Kochi","Tier 2","South","Kerala"),
    ("Nagpur","Tier 3","West","Maharashtra"),
    ("Lucknow","Tier 3","North","Uttar Pradesh"),
    ("Mysore","Tier 3","South","Karnataka"),
    ("Trivandrum","Tier 3","South","Kerala"),
    ("Remote","Remote","Remote","Remote"),
]
df_city_dim = spark.createDataFrame(city_data, schema=city_schema)
print(f"city_dim: {df_city_dim.count()} rows")
df_city_dim.show()


city_dim: 19 rows
+----------+---------+-------+--------------+
|      city|city_tier| region|         state|
+----------+---------+-------+--------------+
| Bangalore|   Tier 1|  South|     Karnataka|
|    Mumbai|   Tier 1|   West|   Maharashtra|
|     Delhi|   Tier 1|  North|         Delhi|
| Hyderabad|   Tier 1|  South|     Telangana|
|   Chennai|   Tier 1|  South|    Tamil Nadu|
|      Pune|   Tier 1|   West|   Maharashtra|
|   Kolkata|   Tier 1|   East|   West Bengal|
| Ahmedabad|   Tier 2|   West|       Gujarat|
|     Noida|   Tier 2|  North| Uttar Pradesh|
|   Gurgaon|   Tier 2|  North|       Haryana|
|    Jaipur|   Tier 2|  North|     Rajasthan|
|    Indore|   Tier 2|Central|Madhya Pradesh|
|Coimbatore|   Tier 2|  South|    Tamil Nadu|
|     Kochi|   Tier 2|  South|        Kerala|
|    Nagpur|   Tier 3|   West|   Maharashtra|
|   Lucknow|   Tier 3|  North| Uttar Pradesh|
|    Mysore|   Tier 3|  South|     Karnataka|
|Trivandrum|   Tier 3|  South|        Kerala|
|    Remote|   R

In [108]:
# DIM 2: Role dimension
role_schema = StructType([
    StructField("job_role",      StringType(), False),
    StructField("role_category", StringType(), False),
    StructField("is_tech",       StringType(), False),
])
role_data = [
    ("Android","Mobile Development","Yes"),("iOS","Mobile Development","Yes"),
    ("Data Science","Data & AI","Yes"),("Machine Learning","Data & AI","Yes"),
    ("Data Engineer","Data & AI","Yes"),("Data Analyst","Data & AI","Yes"),
    ("DevOps","Infrastructure","Yes"),("Cloud","Infrastructure","Yes"),
    ("Backend","Software Engineering","Yes"),("Frontend","Software Engineering","Yes"),
    ("Full Stack","Software Engineering","Yes"),("Java","Software Engineering","Yes"),
    ("Python","Software Engineering","Yes"),("React","Software Engineering","Yes"),
    ("Testing","QA & Testing","Yes"),("QA","QA & Testing","Yes"),
    ("Product Manager","Product & Design","No"),("UI/UX","Product & Design","No"),
    ("HR","Non-Tech","No"),("Finance","Non-Tech","No"),("Sales","Non-Tech","No"),
]
df_role_dim = spark.createDataFrame(role_data, schema=role_schema)
print(f"role_dim: {df_role_dim.count()} rows")
df_role_dim.show()


role_dim: 21 rows
+----------------+--------------------+-------+
|        job_role|       role_category|is_tech|
+----------------+--------------------+-------+
|         Android|  Mobile Development|    Yes|
|             iOS|  Mobile Development|    Yes|
|    Data Science|           Data & AI|    Yes|
|Machine Learning|           Data & AI|    Yes|
|   Data Engineer|           Data & AI|    Yes|
|    Data Analyst|           Data & AI|    Yes|
|          DevOps|      Infrastructure|    Yes|
|           Cloud|      Infrastructure|    Yes|
|         Backend|Software Engineering|    Yes|
|        Frontend|Software Engineering|    Yes|
|      Full Stack|Software Engineering|    Yes|
|            Java|Software Engineering|    Yes|
|          Python|Software Engineering|    Yes|
|           React|Software Engineering|    Yes|
|         Testing|        QA & Testing|    Yes|
|              QA|        QA & Testing|    Yes|
| Product Manager|    Product & Design|     No|
|           UI/UX|    

---
## Stage 2 — Data Cleaning

**Most important step first: rename columns to snake_case.**

Column names with spaces (`Company Name`, `Job Roles`) cause `AnalysisException` in joins, SQL, and many operations.

**Salary conversion:** Raw value `400000` = ₹4,00,000 = **4.0 LPA**. We divide by 100,000.


In [109]:
#  Rename all columns to snake_case 
# This fixes the AnalysisException you saw earlier
df_clean = (
    df_raw
    .withColumnRenamed("Rating",            "rating")
    .withColumnRenamed("Company Name",      "company_name")
    .withColumnRenamed("Job Title",         "job_title")
    .withColumnRenamed("Salary",            "salary_inr")
    .withColumnRenamed("Salaries Reported", "salaries_reported")
    .withColumnRenamed("Location",          "location")
    .withColumnRenamed("Employment Status", "employment_status")
    .withColumnRenamed("Job Roles",         "job_role")
)
print("Renamed columns:", df_clean.columns)


Renamed columns: ['rating', 'company_name', 'job_title', 'salary_inr', 'salaries_reported', 'location', 'employment_status', 'job_role']


In [110]:
# Deduplication 
before = df_clean.count()
df_clean = df_clean.dropDuplicates()
print(f"Dedup: {before - df_clean.count()} rows removed ({before:,} → {df_clean.count():,})")


Dedup: 0 rows removed (22,770 → 22,770)


In [111]:
#  Convert INR → LPA 
# 400000 INR / 100000 = 4.0 LPA (Lakhs Per Annum)
# Standard salary metric in Indian software industry
df_clean = df_clean.withColumn(
    "salary_lpa",
    F.round(F.col("salary_inr") / 100000, 2)
)
print("Sample salary conversion:")
df_clean.select("company_name","salary_inr","salary_lpa").show(5)


Sample salary conversion:
+--------------------+----------+----------+
|        company_name|salary_inr|salary_lpa|
+--------------------+----------+----------+
|              Sasken|    400000|       4.0|
|Advanced Millenni...|    400000|       4.0|
|           Unacademy|   1000000|      10.0|
|Appoids Tech Solu...|    600000|       6.0|
|DXMinds Technologies|    300000|       3.0|
+--------------------+----------+----------+
only showing top 5 rows


In [112]:
#  Null handling 
null_s = df_clean.filter(F.col("salary_lpa").isNull()).count()
print(f"Null salary rows: {null_s} → dropping")
df_clean = df_clean.filter(F.col("salary_lpa").isNotNull())
df_clean = df_clean.filter(F.col("salary_lpa") > 0)

df_clean = df_clean.fillna({
    "rating": 3.5,
    "employment_status": "Full Time",
    "location": "Unknown",
    "job_role": "Other",
})
print("Nulls filled.")


Null salary rows: 0 → dropping
Nulls filled.


In [113]:
#  Outlier removal 
# Salaries outside 1–500 LPA are data entry errors
# They silently distort all averages
before = df_clean.count()
df_clean = df_clean.filter(
    (F.col("salary_lpa") >= 1) & (F.col("salary_lpa") <= 500)
)
print(f"Outliers removed: {before - df_clean.count()} rows")
print(f"Final clean count: {df_clean.count():,} rows")


Outliers removed: 371 rows
Final clean count: 22,399 rows


In [114]:
# Trim + standardise strings 
str_cols = [f.name for f in df_clean.schema.fields
            if isinstance(f.dataType, StringType)]
for c in str_cols:
    df_clean = df_clean.withColumn(c, F.trim(F.col(c)))
df_clean = df_clean.withColumn("company_name", F.initcap(F.col("company_name")))
df_clean = df_clean.withColumn("location",     F.initcap(F.col("location")))
print(f"Trimmed {len(str_cols)} string columns")
df_clean.show(5, truncate=False)


Trimmed 5 string columns
+------+--------------------------------+-----------------+----------+-----------------+---------+-----------------+--------+----------+
|rating|company_name                    |job_title        |salary_inr|salaries_reported|location |employment_status|job_role|salary_lpa|
+------+--------------------------------+-----------------+----------+-----------------+---------+-----------------+--------+----------+
|3.8   |Sasken                          |Android Developer|400000    |3                |Bangalore|Full Time        |Android |4.0       |
|4.5   |Advanced Millennium Technologies|Android Developer|400000    |3                |Bangalore|Full Time        |Android |4.0       |
|4.0   |Unacademy                       |Android Developer|1000000   |3                |Bangalore|Full Time        |Android |10.0      |
|4.4   |Appoids Tech Solutions          |Android Developer|600000    |3                |Bangalore|Full Time        |Android |6.0       |
|3.7   |Dxminds 

---
## Stage 3 — Transformations & Feature Engineering

**7 new derived columns:**
- `salary_band` — LPA quartile buckets for grouping
- `rating_band` — company quality tier
- `is_full_time` — boolean flag (cheap to compute, useful for filters)
- `is_reliable` — salary entries with 5+ reports (statistically trustworthy)
- `weighted_salary` — salary × reports (for weighted avg calculation)
- `company_name_len` — proxy for brand recognition
- `title_seniority` — extracted from free-text job title via regex


In [115]:
df_features = df_clean.withColumn(
    "salary_band",
    F.when(F.col("salary_lpa") < 5,  "Entry  (<5 LPA)")
     .when(F.col("salary_lpa") < 15, "Mid    (5-15 LPA)")
     .when(F.col("salary_lpa") < 30, "Senior (15-30 LPA)")
     .otherwise("Premium (30+ LPA)")
).withColumn(
    "rating_band",
    F.when(F.col("rating") >= 4.0, "Top Rated  (4.0+)")
     .when(F.col("rating") >= 3.0, "Good       (3.0-4.0)")
     .when(F.col("rating") >= 2.0, "Average    (2.0-3.0)")
     .otherwise("Below Avg  (<2.0)")
).withColumn(
    "is_full_time",
    F.col("employment_status") == "Full Time"
).withColumn(
    "is_reliable",
    F.col("salaries_reported") >= 5
).withColumn(
    "weighted_salary",
    F.col("salary_lpa") * F.col("salaries_reported")
).withColumn(
    "company_name_len",
    F.length(F.col("company_name"))
).withColumn(
    "title_seniority",
    F.when(F.lower(F.col("job_title")).rlike("senior|sr\\.|lead|principal|staff"), "Senior")
     .when(F.lower(F.col("job_title")).rlike("junior|jr\\.|associate|entry|intern"), "Junior")
     .when(F.lower(F.col("job_title")).rlike("manager|head|director|vp|chief"), "Management")
     .otherwise("Mid-Level")
)

df_features.select(
    "company_name","job_title","title_seniority",
    "salary_lpa","salary_band","rating","rating_band",
    "is_full_time","is_reliable"
).show(8, truncate=False)


+--------------------------------+-----------------+---------------+----------+-----------------+------+--------------------+------------+-----------+
|company_name                    |job_title        |title_seniority|salary_lpa|salary_band      |rating|rating_band         |is_full_time|is_reliable|
+--------------------------------+-----------------+---------------+----------+-----------------+------+--------------------+------------+-----------+
|Sasken                          |Android Developer|Mid-Level      |4.0       |Entry  (<5 LPA)  |3.8   |Good       (3.0-4.0)|true        |false      |
|Advanced Millennium Technologies|Android Developer|Mid-Level      |4.0       |Entry  (<5 LPA)  |4.5   |Top Rated  (4.0+)   |true        |false      |
|Unacademy                       |Android Developer|Mid-Level      |10.0      |Mid    (5-15 LPA)|4.0   |Top Rated  (4.0+)   |true        |false      |
|Appoids Tech Solutions          |Android Developer|Mid-Level      |6.0       |Mid    (5-15 LP

---
## Stage 4 — Aggregations & Analysis

**HAVING equivalent in PySpark:** `.filter()` AFTER `.agg()` — same concept as SQL HAVING.


In [116]:
#  Salary by job role
print("=== Salary by job role ===")
(
    df_features.groupBy("job_role")
    .agg(
        F.count("*").alias("headcount"),
        F.round(F.avg("salary_lpa"),2).alias("avg_lpa"),
        F.round(F.min("salary_lpa"),2).alias("min_lpa"),
        F.round(F.max("salary_lpa"),2).alias("max_lpa"),
        F.round(F.percentile_approx("salary_lpa",0.5),2).alias("median_lpa"),
    )
    .orderBy("avg_lpa", ascending=False)
    .show(20, truncate=False)
)


=== Salary by job role ===
+--------+---------+-------+-------+-------+----------+
|job_role|headcount|avg_lpa|min_lpa|max_lpa|median_lpa|
+--------+---------+-------+-------+-------+----------+
|Database|865      |9.59   |1.0    |100.0  |7.0       |
|Mobile  |237      |9.1    |1.0    |96.0   |6.0       |
|SDE     |8099     |8.51   |1.0    |98.5   |6.84      |
|Backend |1160     |7.68   |1.0    |52.0   |5.0       |
|IOS     |1613     |7.02   |1.0    |52.0   |6.0       |
|Frontend|2114     |6.23   |1.0    |100.0  |4.56      |
|Android |2870     |5.79   |1.0    |71.0   |4.0       |
|Java    |1838     |5.7    |1.0    |100.0  |4.0       |
|Testing |1725     |5.0    |1.0    |66.0   |4.0       |
|Python  |929      |4.94   |1.0    |69.0   |4.0       |
|Web     |949      |4.34   |1.0    |44.0   |3.12      |
+--------+---------+-------+-------+-------+----------+



In [117]:
#  Salary by company (only those with 10+ reports for reliability)
print("=== Top 15 companies (min 10 salary reports) ===")
(
    df_features.groupBy("company_name")
    .agg(
        F.sum("salaries_reported").alias("total_reports"),
        F.round(F.avg("salary_lpa"),2).alias("avg_lpa"),
        F.round(F.avg("rating"),2).alias("avg_rating"),
    )
    .filter(F.col("total_reports") >= 10)   # HAVING equivalent
    .orderBy("avg_lpa", ascending=False)
    .limit(15)
    .show(truncate=False)
)


=== Top 15 companies (min 10 salary reports) ===
+-------------------+-------------+-------+----------+
|company_name       |total_reports|avg_lpa|avg_rating|
+-------------------+-------------+-------+----------+
|Gojek              |17           |27.08  |4.37      |
|Hotstar            |31           |23.99  |3.3       |
|Cure.fit           |11           |23.5   |4.1       |
|Trilogy Innovations|25           |22.39  |3.5       |
|Media.net          |21           |21.22  |4.2       |
|Sharechat          |46           |20.54  |4.4       |
|Groupon            |32           |20.5   |3.7       |
|Nortonlifelock     |29           |20.38  |3.9       |
|Amd                |12           |18.5   |4.2       |
|Mindtickle         |55           |18.2   |4.7       |
|Inmobi             |56           |17.89  |4.2       |
|Dream11            |46           |17.87  |4.1       |
|Nvidia             |13           |17.45  |4.7       |
|Phonepe            |18           |17.28  |4.24      |
|Blinkit        

In [118]:
# Salary by location (top 10)
print("=== Salary by location (top 10) ===")
(
    df_features.groupBy("location")
    .agg(
        F.count("*").alias("headcount"),
        F.round(F.avg("salary_lpa"),2).alias("avg_lpa"),
        F.round(F.max("salary_lpa"),2).alias("max_lpa"),
    )
    .orderBy("avg_lpa", ascending=False)
    .limit(10)
    .show(truncate=False)
)


=== Salary by location (top 10) ===
+--------------+---------+-------+-------+
|location      |headcount|avg_lpa|max_lpa|
+--------------+---------+-------+-------+
|Mumbai        |746      |9.65   |98.0   |
|Bangalore     |8125     |7.47   |100.0  |
|Kolkata       |175      |7.22   |98.5   |
|Pune          |2087     |7.04   |100.0  |
|Madhya Pradesh|153      |6.85   |78.0   |
|Hyderabad     |4429     |6.84   |97.0   |
|New Delhi     |4073     |6.52   |98.0   |
|Jaipur        |79       |6.44   |25.0   |
|Chennai       |2426     |5.91   |100.0  |
|Kerala        |106      |5.63   |29.0   |
+--------------+---------+-------+-------+



In [119]:
# Salary by title seniority
print("=== Salary by title seniority ===")
(
    df_features.groupBy("title_seniority")
    .agg(
        F.count("*").alias("headcount"),
        F.round(F.avg("salary_lpa"),2).alias("avg_lpa"),
        F.round(F.stddev("salary_lpa"),2).alias("std_dev"),
    )
    .orderBy("avg_lpa", ascending=False)
    .show()
)


=== Salary by title seniority ===
+---------------+---------+-------+-------+
|title_seniority|headcount|avg_lpa|std_dev|
+---------------+---------+-------+-------+
|     Management|       31|   16.3|   10.0|
|         Senior|     2471|  11.14|   8.93|
|      Mid-Level|    17302|   6.87|   6.17|
|         Junior|     2595|   3.96|   3.94|
+---------------+---------+-------+-------+



---
## Stage 5 — Joins

**Join type chosen:**
- Left join for both dimensions — never lose a fact row
- Anti-join — data quality: find locations not in city_dim


In [120]:
# Join city dimension
df_enriched = df_features.join(
    df_city_dim,
    df_features["location"] == df_city_dim["city"],
    how="left"
).drop("city")
print(f"After city join: {df_enriched.count():,} rows")


After city join: 22,399 rows


In [121]:
# Join role dimension
df_enriched = df_enriched.join(
    df_role_dim,
    df_enriched["job_role"] == df_role_dim["job_role"],
    how="left"
).drop(df_role_dim["job_role"])
print(f"After role join: {df_enriched.count():,} rows")
print(f"Columns: {len(df_enriched.columns)}")


After role join: 22,399 rows
Columns: 21


In [122]:
# Anti-join: find unmatched locations (data quality check)
unmatched = df_features.join(
    df_city_dim,
    df_features["location"] == df_city_dim["city"],
    how="left_anti"
)
print(f"Unmatched locations: {unmatched.count()}")
if unmatched.count() > 0:
    print("Add these to city_dim:")
    unmatched.groupBy("location").count().orderBy("count", ascending=False).show(10)


Unmatched locations: 4332
Add these to city_dim:
+--------------+-----+
|      location|count|
+--------------+-----+
|     New Delhi| 4073|
|Madhya Pradesh|  153|
|        Kerala|  106|
+--------------+-----+



In [123]:
# Final preview of enriched data
df_enriched.select(
    "company_name","job_role","role_category","is_tech",
    "salary_lpa","salary_band","location","city_tier","region","rating"
).show(8, truncate=False)


+--------------------------------+--------+------------------+-------+----------+-----------------+---------+---------+------+------+
|company_name                    |job_role|role_category     |is_tech|salary_lpa|salary_band      |location |city_tier|region|rating|
+--------------------------------+--------+------------------+-------+----------+-----------------+---------+---------+------+------+
|Sasken                          |Android |Mobile Development|Yes    |4.0       |Entry  (<5 LPA)  |Bangalore|Tier 1   |South |3.8   |
|Advanced Millennium Technologies|Android |Mobile Development|Yes    |4.0       |Entry  (<5 LPA)  |Bangalore|Tier 1   |South |4.5   |
|Unacademy                       |Android |Mobile Development|Yes    |10.0      |Mid    (5-15 LPA)|Bangalore|Tier 1   |South |4.0   |
|Appoids Tech Solutions          |Android |Mobile Development|Yes    |6.0       |Mid    (5-15 LPA)|Bangalore|Tier 1   |South |4.4   |
|Dxminds Technologies            |Android |Mobile Development|

---
## Stage 6 — Window Functions

**Key difference from groupBy:**
- `groupBy` collapses rows (1 row per group)
- Window keeps all rows, just adds a new computed column

| Function | Behaviour | Use case |
|---|---|---|
| `dense_rank()` | 1,2,2,3 — ties allowed, no gaps | Fair ranking |
| `row_number()` | 1,2,3,4 — always unique | Pick exactly 1 per group |
| `avg().over(window)` | Running average | Trend within partition |


In [124]:
# Rank earners within each job role
window_role = Window.partitionBy("job_role").orderBy(F.desc("salary_lpa"))
df_windowed = df_enriched.withColumn("rank_in_role", F.dense_rank().over(window_role))

print("Top 3 earners per job role:")
(
    df_windowed.filter(F.col("rank_in_role") <= 3)
    .select("job_role","rank_in_role","company_name","job_title","salary_lpa","location")
    .orderBy("job_role","rank_in_role")
    .show(20, truncate=False)
)


Top 3 earners per job role:
+--------+------------+-------------------------+----------------------------------------------+----------+---------+
|job_role|rank_in_role|company_name             |job_title                                     |salary_lpa|location |
+--------+------------+-------------------------+----------------------------------------------+----------+---------+
|Android |1           |Fp Tech Science          |Android Developer                             |71.0      |Bangalore|
|Android |2           |Conzumex Industries      |Android Developer                             |69.0      |Bangalore|
|Android |3           |Neutrino Tech Systems    |Android Developer                             |50.0      |Pune     |
|Backend |1           |Gitlab                   |Senior Backend Engineer                       |52.0      |Bangalore|
|Backend |2           |Appincubator             |Senior Backend Engineer                       |43.0      |Hyderabad|
|Backend |3           |Inovo

In [125]:
# Row number: top earner per city (unique winner)
window_city = Window.partitionBy("location").orderBy(F.desc("salary_lpa"))
df_windowed = df_windowed.withColumn("rank_in_city", F.row_number().over(window_city))

print("Top earner per city:")
(
    df_windowed.filter(F.col("rank_in_city") == 1)
    .select("location","company_name","job_title","salary_lpa","rating")
    .orderBy("salary_lpa", ascending=False)
    .show(15, truncate=False)
)


Top earner per city:
+--------------+----------------------+--------------------------------------------+----------+------+
|location      |company_name          |job_title                                   |salary_lpa|rating|
+--------------+----------------------+--------------------------------------------+----------+------+
|Bangalore     |Concentrix            |Oracle Database Administrator               |100.0     |3.8   |
|Chennai       |Oasys Cybernetics     |Senior Java Developer                       |100.0     |3.6   |
|Pune          |Koru Ux Design        |Senior Front End Developer                  |100.0     |3.5   |
|Kolkata       |Amazon                |Software Development Engineer (SDE)         |98.5      |3.8   |
|Mumbai        |Fff Enterprises       |Non Software Development Engineer           |98.0      |4.2   |
|New Delhi     |Digital Raju          |Software Development Engineer (SDE)         |98.0      |4.3   |
|Hyderabad     |Gaana Lyrics Point.com|Software Deve

In [126]:
# % of role max salary
window_pct = Window.partitionBy("job_role")
df_windowed = df_windowed.withColumn(
    "pct_of_role_max",
    F.round(F.col("salary_lpa") / F.max("salary_lpa").over(window_pct) * 100, 1)
)

# Running average salary within each role
window_run = (
    Window.partitionBy("job_role").orderBy("salary_lpa")
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)
df_windowed = df_windowed.withColumn(
    "running_avg_lpa",
    F.round(F.avg("salary_lpa").over(window_run), 2)
)

df_windowed.select(
    "job_role","company_name","salary_lpa",
    "rank_in_role","pct_of_role_max","running_avg_lpa"
).show(10, truncate=False)


+--------+------------------------+----------+------------+---------------+---------------+
|job_role|company_name            |salary_lpa|rank_in_role|pct_of_role_max|running_avg_lpa|
+--------+------------------------+----------+------------+---------------+---------------+
|Android |Varadhi Smartek         |1.0       |131         |1.4            |1.0            |
|Android |Dada                    |1.0       |131         |1.4            |1.0            |
|Android |Neutrinos Solutions     |1.0       |131         |1.4            |1.0            |
|Android |Pragmatic Solutions     |1.0       |131         |1.4            |1.0            |
|Android |Sixhops                 |1.0       |131         |1.4            |1.0            |
|Android |Infoskaters Technologies|1.0       |131         |1.4            |1.0            |
|Android |Ss Consulting           |1.0       |131         |1.4            |1.0            |
|Android |Igrand It Solutions     |1.0       |131         |1.4            |1.0  

---
## Stage 7 — Performance Optimisation

| Technique | What it does | When to use |
|---|---|---|
| `cache()` | Store in memory after first compute | Use DF 2+ times |
| `repartition(n, col)` | Full shuffle, even distribution | Before heavy agg/join |
| `coalesce(n)` | Merge partitions, no shuffle | Before writing files |
| `explain()` | Show physical execution plan | Diagnosing slow jobs |


In [127]:
print(f"Partitions before: {df_windowed.rdd.getNumPartitions()}")

# Repartition by job_role — most groupBys use this column
df_repartitioned = df_windowed.repartition(4, "job_role")
print(f"After repartition(4, job_role): {df_repartitioned.rdd.getNumPartitions()}")

# Cache: reused in Stage 8 (write) + Stage 9 (SQL)
df_cached = df_repartitioned.cache()
print(f"Cached: {df_cached.count():,} rows")

# Coalesce before write: no shuffle, fewer output files
df_write_ready = df_cached.coalesce(2)
print(f"After coalesce(2): {df_write_ready.rdd.getNumPartitions()} partitions")


Partitions before: 1
After repartition(4, job_role): 4
Cached: 22,399 rows
After coalesce(2): 2 partitions


---
## Stage 8 — Data Storage

**Why Parquet?**
- Columnar: `SELECT salary_lpa` reads only that column, not all 20
- Compressed: ~75% smaller than CSV
- Partition pruning: `WHERE job_role='Android'` reads only the Android folder

**Partition key choice:** `job_role` + `employment_status` — analysts always filter by these.


In [128]:
#   STAGE 8 — COMPLETE (PANDAS VERSION)
import os
import pandas as pd

output_base = os.path.abspath("./output")
os.makedirs(output_base, exist_ok=True)

#  Convert full dataset to pandas 
print("Converting to pandas... please wait")
df_pandas = df_write_ready.toPandas()
print(f"Converted: {len(df_pandas):,} rows x {len(df_pandas.columns)} columns")

#  Save full enriched dataset 
full_csv = os.path.join(output_base, "glassdoor_enriched_full.csv")
df_pandas.to_csv(full_csv, index=False)
print(f"[OK] Full dataset → {full_csv}")
print(f"     Size: {os.path.getsize(full_csv)/1024/1024:.1f} MB")

#  Build and save summary ─
summary = (
    df_pandas
    .groupby([
        "job_role", "location",
        "employment_status", "salary_band",
        "title_seniority"
    ])
    .agg(
        headcount  = ("salary_lpa", "count"),
        avg_lpa    = ("salary_lpa", "mean"),
        max_lpa    = ("salary_lpa", "max"),
        avg_rating = ("rating",     "mean"),
    )
    .round(2)
    .reset_index()
    .sort_values(
        ["job_role", "avg_lpa"],
        ascending=[True, False]
    )
)
summary_csv = os.path.join(output_base, "glassdoor_summary.csv")
summary.to_csv(summary_csv, index=False)
print(f"[OK] Summary → {summary_csv}")
print(f"     Rows: {len(summary):,}")

#  Verify 
print(f"\n[VERIFY]")
# Read back and confirm
df_check = pd.read_csv(full_csv)
print(f"  Full CSV rows    : {len(df_check):,}")
print(f"  Full CSV columns : {len(df_check.columns)}")
print(f"  Columns          : {list(df_check.columns)}")

print(f"\n[FILES IN OUTPUT FOLDER]")
for f in os.listdir(output_base):
    fpath = os.path.join(output_base, f)
    size  = os.path.getsize(fpath) / 1024 / 1024
    print(f"  {f:45s}  {size:.1f} MB")

print("\n[STAGE 8 COMPLETE] ")

Converting to pandas... please wait
Converted: 22,399 rows x 25 columns
[OK] Full dataset → c:\Users\Lenovo\AppData\Local\Temp\1a27026f-d723-4257-876f-fe71bc3a24e1_files.zip.4e1\output\glassdoor_enriched_full.csv
     Size: 4.2 MB
[OK] Summary → c:\Users\Lenovo\AppData\Local\Temp\1a27026f-d723-4257-876f-fe71bc3a24e1_files.zip.4e1\output\glassdoor_summary.csv
     Rows: 583

[VERIFY]
  Full CSV rows    : 22,399
  Full CSV columns : 25
  Columns          : ['rating', 'company_name', 'job_title', 'salary_inr', 'salaries_reported', 'location', 'employment_status', 'job_role', 'salary_lpa', 'salary_band', 'rating_band', 'is_full_time', 'is_reliable', 'weighted_salary', 'company_name_len', 'title_seniority', 'city_tier', 'region', 'state', 'role_category', 'is_tech', 'rank_in_role', 'rank_in_city', 'pct_of_role_max', 'running_avg_lpa']

[FILES IN OUTPUT FOLDER]
  glassdoor_enriched_full.csv                    4.2 MB
  glassdoor_enriched_parquet                     0.0 MB
  glassdoor_summary.

In [129]:
# Use df_pandas which we already created in Stage 8
# Finding 1 — Top paying role
print("=== TOP PAYING ROLES ===")
top_roles = (
    df_pandas
    .groupby("job_role")["salary_lpa"]
    .mean()
    .round(1)
    .sort_values(ascending=False)
    .head(3)
    .reset_index()
)
top_roles.columns = ["job_role", "avg_lpa"]
print(top_roles.to_string(index=False))

# Finding 2 — Top paying city
print("\n=== TOP PAYING CITIES ===")
top_cities = (
    df_pandas
    .groupby("location")["salary_lpa"]
    .mean()
    .round(1)
    .sort_values(ascending=False)
    .head(3)
    .reset_index()
)
top_cities.columns = ["location", "avg_lpa"]
print(top_cities.to_string(index=False))

# Finding 3 — Seniority gap
print("\n=== SALARY BY SENIORITY ===")
seniority = (
    df_pandas
    .groupby("title_seniority")["salary_lpa"]
    .mean()
    .round(1)
    .sort_values(ascending=False)
    .reset_index()
)
seniority.columns = ["title_seniority", "avg_lpa"]
print(seniority.to_string(index=False))

=== TOP PAYING ROLES ===
job_role  avg_lpa
Database      9.6
  Mobile      9.1
     SDE      8.5

=== TOP PAYING CITIES ===
 location  avg_lpa
   Mumbai      9.6
Bangalore      7.5
  Kolkata      7.2

=== SALARY BY SENIORITY ===
title_seniority  avg_lpa
     Management     16.3
         Senior     11.1
      Mid-Level      6.9
         Junior      4.0


---
## Stage 9 — Spark SQL

`createOrReplaceTempView()` registers a DataFrame as a virtual SQL table — zero data copy.


In [130]:
# SQL ANALYSIS (TEMP VIEW)
df_cached.createOrReplaceTempView("salary_data")
print("Temp view registered: salary_data")
print("Columns:", df_cached.columns)


Temp view registered: salary_data
Columns: ['rating', 'company_name', 'job_title', 'salary_inr', 'salaries_reported', 'location', 'employment_status', 'job_role', 'salary_lpa', 'salary_band', 'rating_band', 'is_full_time', 'is_reliable', 'weighted_salary', 'company_name_len', 'title_seniority', 'city_tier', 'region', 'state', 'role_category', 'is_tech', 'rank_in_role', 'rank_in_city', 'pct_of_role_max', 'running_avg_lpa']


In [131]:
# Example SQL query: top 10 job roles by average salary (min 5 reports for reliability)
spark.sql("""
    SELECT job_role, COUNT(*) AS headcount,
           ROUND(AVG(salary_lpa),2) AS avg_lpa,
           ROUND(MAX(salary_lpa),2) AS max_lpa
    FROM salary_data
    GROUP BY job_role
    HAVING COUNT(*) >= 5
    ORDER BY avg_lpa DESC
    LIMIT 10
""").show(truncate=False)


+--------+---------+-------+-------+
|job_role|headcount|avg_lpa|max_lpa|
+--------+---------+-------+-------+
|Database|865      |9.59   |100.0  |
|Mobile  |237      |9.1    |96.0   |
|SDE     |8099     |8.51   |98.5   |
|Backend |1160     |7.68   |52.0   |
|IOS     |1613     |7.02   |52.0   |
|Frontend|2114     |6.23   |100.0  |
|Android |2870     |5.79   |71.0   |
|Java    |1838     |5.7    |100.0  |
|Testing |1725     |5.0    |66.0   |
|Python  |929      |4.94   |69.0   |
+--------+---------+-------+-------+



In [132]:
# Top 5 companies per city using window in SQL
spark.sql("""
    SELECT location, company_name, avg_lpa, avg_rating, headcount
    FROM (
        SELECT location, company_name,
               ROUND(AVG(salary_lpa),2) AS avg_lpa,
               ROUND(AVG(rating),2) AS avg_rating,
               COUNT(*) AS headcount,
               DENSE_RANK() OVER (
                   PARTITION BY location ORDER BY AVG(salary_lpa) DESC
               ) AS rnk
        FROM salary_data
        WHERE location IN ('Bangalore','Mumbai','Hyderabad','Pune','Chennai')
        GROUP BY location, company_name
        HAVING COUNT(*) >= 3
    )
    WHERE rnk <= 5
    ORDER BY location, avg_lpa DESC
""").show(30, truncate=False)


+---------+-------------------+-------+----------+---------+
|location |company_name       |avg_lpa|avg_rating|headcount|
+---------+-------------------+-------+----------+---------+
|Bangalore|Fp Tech Science    |34.67  |4.4       |3        |
|Bangalore|Gojek              |27.73  |4.37      |11       |
|Bangalore|Conzumex Industries|27.12  |4.1       |3        |
|Bangalore|Trilogy Innovations|26.35  |3.5       |5        |
|Bangalore|Cure.fit           |24.2   |4.1       |5        |
|Chennai  |Oasys Cybernetics  |27.07  |3.6       |4        |
|Chennai  |Amazon Lab126      |24.0   |3.7       |3        |
|Chennai  |Bank Of America    |16.19  |4.0       |5        |
|Chennai  |Walmart Global Tech|15.28  |4.0       |3        |
|Chennai  |Nortonlifelock     |15.11  |3.9       |8        |
|Hyderabad|Joveo              |26.25  |4.6       |4        |
|Hyderabad|Tide               |24.33  |4.0       |3        |
|Hyderabad|Broadcom           |21.25  |3.5       |4        |
|Hyderabad|Flipkart     

In [133]:
# CTE: seniority pay gap
spark.sql("""
    WITH seniority_avg AS (
        SELECT job_role, title_seniority,
               ROUND(AVG(salary_lpa),2) AS avg_lpa, COUNT(*) AS cnt
        FROM salary_data GROUP BY job_role, title_seniority HAVING COUNT(*) >= 3
    ),
    pivoted AS (
        SELECT job_role,
               MAX(CASE WHEN title_seniority='Senior'    THEN avg_lpa END) AS senior_lpa,
               MAX(CASE WHEN title_seniority='Mid-Level' THEN avg_lpa END) AS mid_lpa,
               MAX(CASE WHEN title_seniority='Junior'    THEN avg_lpa END) AS junior_lpa
        FROM seniority_avg GROUP BY job_role
    )
    SELECT job_role, senior_lpa, mid_lpa, junior_lpa,
           ROUND(senior_lpa - junior_lpa, 2) AS senior_junior_gap
    FROM pivoted
    WHERE senior_lpa IS NOT NULL AND junior_lpa IS NOT NULL
    ORDER BY senior_junior_gap DESC
""").show(15, truncate=False)


+--------+----------+-------+----------+-----------------+
|job_role|senior_lpa|mid_lpa|junior_lpa|senior_junior_gap|
+--------+----------+-------+----------+-----------------+
|Backend |15.4      |7.73   |3.09      |12.31            |
|Frontend|14.63     |5.86   |3.49      |11.14            |
|SDE     |12.75     |8.81   |4.52      |8.23             |
|Database|12.91     |8.59   |4.75      |8.16             |
|Android |10.46     |5.29   |3.01      |7.45             |
|IOS     |9.99      |6.39   |3.8       |6.19             |
|Python  |8.56      |5.0    |3.03      |5.53             |
|Java    |8.98      |5.0    |3.66      |5.32             |
|Mobile  |15.74     |7.45   |14.54     |1.2              |
|Testing |5.35      |4.94   |5.43      |-0.08            |
+--------+----------+-------+----------+-----------------+



In [134]:
# Reliability-filtered salary benchmark with percentiles
spark.sql("""
    SELECT job_role, title_seniority, COUNT(*) AS headcount,
           ROUND(AVG(salary_lpa),2) AS avg_lpa,
           ROUND(PERCENTILE_APPROX(salary_lpa,0.25),2) AS p25,
           ROUND(PERCENTILE_APPROX(salary_lpa,0.50),2) AS median,
           ROUND(PERCENTILE_APPROX(salary_lpa,0.75),2) AS p75
    FROM salary_data
    WHERE is_reliable = true
    GROUP BY job_role, title_seniority
    HAVING COUNT(*) >= 3
    ORDER BY avg_lpa DESC
""").show(20, truncate=False)


+--------+---------------+---------+-------+----+------+----+
|job_role|title_seniority|headcount|avg_lpa|p25 |median|p75 |
+--------+---------------+---------+-------+----+------+----+
|Testing |Management     |4        |10.75  |4.0 |12.0  |12.0|
|SDE     |Senior         |75       |8.9    |5.0 |7.0   |12.0|
|Backend |Mid-Level      |8        |8.24   |1.92|5.0   |9.0 |
|SDE     |Mid-Level      |532      |8.18   |3.0 |6.0   |12.0|
|SDE     |Junior         |52       |4.66   |3.12|4.68  |6.0 |
|Testing |Senior         |50       |4.47   |3.0 |4.0   |6.0 |
|Java    |Mid-Level      |30       |3.61   |2.0 |3.0   |4.08|
|Frontend|Mid-Level      |7        |3.35   |2.0 |3.0   |4.0 |
|Java    |Senior         |14       |3.14   |2.0 |3.0   |4.0 |
|Testing |Mid-Level      |148      |3.02   |2.0 |2.4   |4.0 |
|Web     |Mid-Level      |21       |2.9    |2.0 |2.4   |3.0 |
|Java    |Junior         |20       |2.4    |1.44|1.92  |3.0 |
|Python  |Mid-Level      |14       |2.14   |2.0 |2.0   |2.52|
+-------

---
## Stage 10 — Pipeline Summary

In [135]:
print("=" * 55)
print("  GLASSDOOR SALARY ETL — COMPLETE")
print("=" * 55)
print(f"  Raw rows ingested     : {df_raw.count():>8,}")
print(f"  After cleaning        : {df_clean.count():>8,}")
print(f"  Features added        : {len(df_features.columns)-len(df_clean.columns):>8}")
print(f"  Enriched columns      : {len(df_enriched.columns):>8}")
print(f"  Window cols added     : {len(df_windowed.columns)-len(df_enriched.columns):>8}")
print(f"  Parquet partition key : job_role x employment_status")
print(f"  Spark SQL view        : salary_data")
print(f"  Output location       : ./output/")
print("=" * 55)
spark.stop()


  GLASSDOOR SALARY ETL — COMPLETE
  Raw rows ingested     :   22,770
  After cleaning        :   22,399
  Features added        :        7
  Enriched columns      :       21
  Window cols added     :        4
  Parquet partition key : job_role x employment_status
  Spark SQL view        : salary_data
  Output location       : ./output/


In [136]:
# Top paying role
df_cached.groupBy("job_role") \
    .agg(F.round(F.avg("salary_lpa"),1).alias("avg_lpa")) \
    .orderBy("avg_lpa", ascending=False).show(3)

# Top paying city
df_cached.groupBy("location") \
    .agg(F.round(F.avg("salary_lpa"),1).alias("avg_lpa")) \
    .orderBy("avg_lpa", ascending=False).show(3)

# Seniority gap
df_cached.groupBy("title_seniority") \
    .agg(F.round(F.avg("salary_lpa"),1).alias("avg_lpa")) \
    .orderBy("avg_lpa", ascending=False).show()

PySparkRuntimeError: [SESSION_OR_CONTEXT_NOT_EXISTS] SparkContext or SparkSession should be created first.

---
## Resume Bullet Points

**Project: Glassdoor India Software Salary Analytics Pipeline | PySpark, Python, Parquet**

- Built end-to-end ETL pipeline on 22,000+ Glassdoor salary records using PySpark, resolving `AnalysisException` caused by space-delimited column names through systematic snake_case renaming
- Converted raw Indian Rupee salary integers to LPA (Lakhs Per Annum) industry standard and engineered 7 derived features including salary bands, reliability scores, and NLP-based seniority extraction via regex
- Designed star schema with city-tier and role-category dimension tables; enriched fact table via left joins and detected unmapped location codes using anti-joins for upstream data quality reporting
- Ranked top earners per job role and city using `dense_rank` and `row_number` window functions, and computed running averages and salary percentile position columns
- Optimised pipeline through column-partitioned repartitioning, strategic caching to eliminate redundant DAG re-execution, and pre-write coalescing
- Persisted enriched dataset as partitioned Parquet (by job role and employment status) enabling partition pruning; generated BI-ready CSV summary for Tableau/Power BI consumption
- Executed 5 Spark SQL analytics queries including CTE-based seniority pay gap analysis, window-ranked top companies per city, and percentile salary benchmarks on reliability-filtered data
